In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import random

# Set seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [2]:

# =============================================================================
# LSTM WIND SPEED FORECASTING — LAG LENGTH OPTIMISATION (1 to 14)
# =============================================================================
# Univariate wind speed forecasting (WS10M) using a Long Short-Term Memory
# (LSTM) network. The optimal autoregressive lag length (1–14) is identified
# jointly with model hyperparameters under four strategies:
#
#   1. Original Model    (manual hyperparameters — tested across lags 1–14)
#   2. Grid Search       (exhaustive grid over hyperparams × lags 1–14)
#   3. Randomized Search (random sampling over hyperparams × lags 1–14)
#   4. Optuna            (Bayesian optimisation over hyperparams + lag)
#
# Data split  : 80 % train | 20 % test (chronological, no shuffling).
# CV strategy : TimeSeriesSplit applied exclusively on the training set.
# The test set is used ONLY for final evaluation — never during tuning.
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1: PACKAGE INSTALLATION
# ─────────────────────────────────────────────────────────────────────────────

! pip install optuna
# pip install tensorflow
# pip install scikit-learn



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 6.4 MB/s eta 0:00:00


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2: IMPORT LIBRARIES
# ─────────────────────────────────────────────────────────────────────────────

# Standard libraries
import random
import warnings
warnings.filterwarnings("ignore")

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Scikit-learn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Optuna for Bayesian optimisation
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)   # Suppress verbose output

In [4]:

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3: REPRODUCIBILITY SEEDS
# ─────────────────────────────────────────────────────────────────────────────

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4: LOAD DATASET
# ─────────────────────────────────────────────────────────────────────────────

# Dataset source (Google Sheets exported as CSV):
# https://docs.google.com/spreadsheets/d/1j_Euo80PrGckVDVr2hTG9zZebxJD0TSC

sheet_id   = "1j_Euo80PrGckVDVr2hTG9zZebxJD0TSC"
sheet_name = "Sheet1"
csv_url    = (
    f"https://docs.google.com/spreadsheets/d/{sheet_id}"
    f"/gviz/tq?tqx=out:csv&sheet={sheet_name}"
)

# Read the Google Sheet as CSV
df = pd.read_csv(csv_url)

# Backup original DataFrame before any modifications
df_backup = df.copy()

# Set Date as the index and convert to datetime for proper date-axis plotting
df = df.set_index('Date')
df.index = pd.to_datetime(df.index)

# Drop calendar columns (YEAR, MO, DY) — not needed as model features
df = df.drop(columns=["YEAR", "MO", "DY"])

# Keep only the target variable: Wind Speed at 10 metres (WS10M)
df = df[["WS10M"]].astype(float)

print("Dataset loaded. Shape:", df.shape)
print(df.head())

Dataset loaded. Shape: (4017, 1)
            WS10M
Date             
2013-01-01   3.99
2013-01-02   4.25
2013-01-03   4.75
2013-01-04   5.74
2013-01-05   5.79


In [5]:

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5: SHARED HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def create_lagged_features(data, lags):
    """
    Create autoregressive lag features for the time series.

    Parameters:
        data : DataFrame containing the WS10M column
        lags : int, number of lag periods to create

    Returns:
        lagged_df : DataFrame with original column plus lag columns,
                    NaN rows dropped
    """
    lagged_df = data.copy()
    for lag in range(1, lags + 1):
        lagged_df[f'lag_{lag}'] = lagged_df['WS10M'].shift(lag)
    return lagged_df.dropna()


def prepare_lag_data(lag):
    """
    Build lag features, split 80/20 chronologically, and apply MinMaxScaling.

    NOTE: Both scalers are fit ONLY on the training portion to prevent
    data leakage from the test set into the training transform.

    Parameters:
        lag : int, number of autoregressive lag terms

    Returns:
        X_train_lstm   : numpy array, shape (n_train, lag, 1) — LSTM input
        X_test_lstm    : numpy array, shape (n_test,  lag, 1) — LSTM input
        y_train_scaled : numpy array, scaled training target (flat)
        y_test_scaled  : numpy array, scaled test target (flat)
        y_test_inv     : numpy array, test target in original scale
        y_test_raw     : pandas Series, raw test target (for metrics)
        test_dates     : DatetimeIndex, dates of test observations
        scaler_y       : fitted MinMaxScaler for inverse transform
    """
    df_lagged  = create_lagged_features(df, lag)

    # Chronological 80/20 split
    train_size = int(len(df_lagged) * 0.8)
    train      = df_lagged.iloc[:train_size]
    test       = df_lagged.iloc[train_size:]
    test_dates = test.index

    X_train = train.drop(columns=['WS10M'])
    y_train = train['WS10M']
    X_test  = test.drop(columns=['WS10M'])
    y_test  = test['WS10M']

    # Fit scalers on training data ONLY — transform test with same scaler
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled  = scaler_X.transform(X_test)

    y_train_scaled = scaler_y.fit_transform(
        y_train.values.reshape(-1, 1)
    ).flatten()
    y_test_scaled  = scaler_y.transform(
        y_test.values.reshape(-1, 1)
    ).flatten()

    # Inverse-scale the test target for metric computation
    y_test_inv = scaler_y.inverse_transform(
        y_test_scaled.reshape(-1, 1)
    ).flatten()

    # Reshape for LSTM input: (samples, timesteps=lag, features=1)
    X_train_lstm = X_train_scaled.reshape(
        (X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
    )
    X_test_lstm  = X_test_scaled.reshape(
        (X_test_scaled.shape[0],  X_test_scaled.shape[1],  1)
    )

    return (X_train_lstm, X_test_lstm,
            y_train_scaled, y_test_scaled,
            y_test_inv, y_test, test_dates, scaler_y)


def build_lstm(n_units, dropout_rate, learning_rate, input_shape):
    """
    Build and compile a single-layer LSTM model for regression.

    Parameters:
        n_units       : int,   number of LSTM units
        dropout_rate  : float, dropout rate after LSTM layer
        learning_rate : float, Adam learning rate
        input_shape   : tuple, (timesteps, features)

    Returns:
        compiled Keras Sequential model
    """
    model = Sequential([
        LSTM(n_units, activation='tanh', input_shape=input_shape),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(
        optimizer = Adam(learning_rate=learning_rate),
        loss      = 'mean_squared_error'
    )
    return model


def cross_validate_lstm(n_units, dropout_rate, learning_rate,
                        batch_size, n_epochs,
                        X_train_lstm, y_train_scaled,
                        n_splits=3):
    """
    Evaluate an LSTM configuration using TimeSeriesSplit CV on training data.
    Returns mean validation MSE, or infinity on error.

    Parameters:
        n_units, dropout_rate, learning_rate : LSTM hyperparameters
        batch_size, n_epochs                 : training hyperparameters
        X_train_lstm  : numpy array, (n_train, timesteps, 1)
        y_train_scaled: numpy array, scaled training target
        n_splits      : int, number of TimeSeriesSplit folds

    Returns:
        float, mean validation MSE across folds (lower is better)
    """
    tscv       = TimeSeriesSplit(n_splits=n_splits)
    val_losses = []

    try:
        for train_idx, val_idx in tscv.split(X_train_lstm):
            X_t, X_v = X_train_lstm[train_idx], X_train_lstm[val_idx]
            y_t, y_v = y_train_scaled[train_idx], y_train_scaled[val_idx]

            model = build_lstm(
                n_units, dropout_rate, learning_rate,
                input_shape=X_t.shape[1:]
            )
            model.fit(
                X_t, y_t,
                epochs     = n_epochs,
                batch_size = batch_size,
                verbose    = 0
            )
            pred     = model.predict(X_v, verbose=0).flatten()
            val_losses.append(mean_squared_error(y_v, pred))

    except Exception as e:
        print(f"    Skipping combination due to error: {e}")
        return float('inf')

    return np.mean(val_losses)


def compute_metrics(y_true, y_pred):
    """
    Compute MSE, RMSE, MAE, MAPE, and R² for a set of predictions.

    Parameters:
        y_true : array-like, actual values (original scale)
        y_pred : array-like, predicted values (original scale)

    Returns:
        dict with keys: mse, rmse, mae, mape, r2
    """
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()

    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    # Small epsilon in denominator prevents division by zero
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8))) * 100
    r2   = r2_score(y_true, y_pred)

    return dict(mse=mse, rmse=rmse, mae=mae, mape=mape, r2=r2)


def print_metrics(metrics, label):
    """Print a formatted metrics report."""
    print(f"\n{'*' * 55}")
    print(f"  {label}")
    print(f"{'*' * 55}")
    print(f"  MSE       : {metrics['mse']:.6f}")
    print(f"  RMSE      : {metrics['rmse']:.6f}")
    print(f"  MAE       : {metrics['mae']:.6f}")
    print(f"  MAPE      : {metrics['mape']:.6f} %")
    print(f"  R-squared : {metrics['r2']:.6f}")
    print(f"{'*' * 55}")


def print_optimal_model_box(method_name, optimal_lag, best_params, metrics):
    """
    Print a clearly formatted box summarising the optimal model found
    by a given hyperparameter tuning method.

    Parameters:
        method_name : str,  e.g. 'Original Model', 'Grid Search'
        optimal_lag : int,  best lag length identified
        best_params : dict, hyperparameters of the optimal model
        metrics     : dict, keys mse / rmse / mae / mape / r2
    """
    border = "█" * 65
    print(f"\n{border}")
    print(f"  ✦  OPTIMAL MODEL SUMMARY — {method_name.upper()}")
    print(border)
    print(f"  Optimal Lag Length : {optimal_lag}")
    print(f"  {'─' * 61}")
    print(f"  Hyperparameters:")
    for k, v in best_params.items():
        print(f"    {k:<22} : {v}")
    print(f"  {'─' * 61}")
    print(f"  Test Set Performance:")
    print(f"    {'MSE':<22} : {metrics['mse']:.6f}")
    print(f"    {'RMSE':<22} : {metrics['rmse']:.6f}")
    print(f"    {'MAE':<22} : {metrics['mae']:.6f}")
    print(f"    {'MAPE':<22} : {metrics['mape']:.6f} %")
    print(f"    {'R-squared':<22} : {metrics['r2']:.6f}")
    print(f"{border}\n")


def plot_predictions(test_dates, y_true, y_pred, title):
    """
    Plot actual vs predicted wind speed values against date.

    Parameters:
        test_dates : DatetimeIndex
        y_true     : array-like, actual values
        y_pred     : array-like, predicted values
        title      : str, plot title
    """
    plt.figure(figsize=(14, 6))
    plt.plot(test_dates, y_true, label="Actual",    linewidth=2,   linestyle="-")
    plt.plot(test_dates, y_pred, label="Predicted", linewidth=1.5, linestyle="--")
    plt.title(title, fontsize=14)
    plt.xlabel("Date", fontsize=12)
    plt.ylabel("Wind Speed (WS10M)", fontsize=12)
    plt.gca().xaxis.set_major_formatter(DateFormatter("%Y-%m"))
    plt.gcf().autofmt_xdate()
    plt.legend(fontsize=12)
    plt.grid(linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6: ORIGINAL LSTM MODEL (Manual Hyperparameters, Lags 1–14)
# ─────────────────────────────────────────────────────────────────────────────
# Fixed hyperparameters are used across all lag lengths. The lag that produces
# the lowest test RMSE is selected as the optimal window size.

print("\n" + "=" * 70)
print("SECTION 6: ORIGINAL LSTM MODEL  (No Hyperparameter Tuning)")
print("=" * 70)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Fixed hyperparameters for the baseline LSTM
ORIG_PARAMS = {
    'n_units':       100,
    'dropout_rate':  0.2,
    'learning_rate': 0.001,
    'batch_size':    32,
    'n_epochs':      50,
}

orig_results = []

for lag in range(1, 15):
    print(f"\n{'=' * 50}")
    print(f"  Original LSTM — Lag {lag}")
    print(f"{'=' * 50}")

    (X_train_lstm, X_test_lstm,
     y_train_scaled, y_test_scaled,
     y_test_inv, y_test_raw,
     test_dates, scaler_y) = prepare_lag_data(lag)

    # Train on the full training set with fixed parameters
    model_orig = build_lstm(
        ORIG_PARAMS['n_units'],
        ORIG_PARAMS['dropout_rate'],
        ORIG_PARAMS['learning_rate'],
        input_shape = X_train_lstm.shape[1:]
    )
    model_orig.fit(
        X_train_lstm, y_train_scaled,
        epochs     = ORIG_PARAMS['n_epochs'],
        batch_size = ORIG_PARAMS['batch_size'],
        verbose    = 0
    )

    # Predict and inverse scale
    y_pred_scaled = model_orig.predict(X_test_lstm, verbose=0).flatten()
    y_pred_inv    = scaler_y.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).flatten()

    # Compute and print per-lag metrics
    m = compute_metrics(y_test_inv, y_pred_inv)
    print_metrics(m, f"Original LSTM — Lag {lag}")

    # Plot actual vs predicted
    plot_predictions(
        test_dates, y_test_inv, y_pred_inv,
        title=f"Original LSTM — Lag {lag}: Actual vs Predicted Wind Speed"
    )

    orig_results.append({'lag': lag, **m})

# Identify optimal lag for the original model
orig_summary  = pd.DataFrame(orig_results).set_index('lag')
best_orig_lag = orig_summary['rmse'].idxmin()
best_orig_row = orig_summary.loc[best_orig_lag].to_dict()

print("\n" + "=" * 70)
print("ORIGINAL LSTM — SUMMARY ACROSS ALL LAGS")
print("=" * 70)
print(orig_summary.round(6))

# ── Optimal model box ────────────────────────────────────────────────────────
print_optimal_model_box(
    method_name = "Original LSTM",
    optimal_lag = best_orig_lag,
    best_params = ORIG_PARAMS,
    metrics     = best_orig_row
)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7: GRID SEARCH — HYPERPARAMETERS × LAGS 1–14
# ─────────────────────────────────────────────────────────────────────────────
# For each lag (1–14), an exhaustive grid search is conducted over the
# hyperparameter space using TimeSeriesSplit CV on the training set.
# The test set is NEVER accessed during the search.

print("\n" + "=" * 70)
print("SECTION 7: GRID SEARCH HYPERPARAMETER TUNING")
print("=" * 70)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Hyperparameter search grid
PARAM_GRID = {
    'n_units':       [32, 64],
    'dropout_rate':  [0.1, 0.2],
    'learning_rate': [0.001, 0.01],
    'batch_size':    [32],
    'n_epochs':      [20, 30]
}

grid_results = []

for lag in range(1, 15):
    print(f"\n{'=' * 50}")
    print(f"  Grid Search — Lag {lag}")
    print(f"{'=' * 50}")

    (X_train_lstm, X_test_lstm,
     y_train_scaled, y_test_scaled,
     y_test_inv, y_test_raw,
     test_dates, scaler_y) = prepare_lag_data(lag)

    best_val_loss_grid = float('inf')
    best_params_grid   = None

    # Calculate total number of combinations for progress display
    total = (len(PARAM_GRID['n_units']) *
             len(PARAM_GRID['dropout_rate']) *
             len(PARAM_GRID['learning_rate']) *
             len(PARAM_GRID['batch_size']) *
             len(PARAM_GRID['n_epochs']))
    combo = 0

    # Exhaustive grid search using TimeSeriesSplit CV on training data only
    for n_units in PARAM_GRID['n_units']:
        for dropout_rate in PARAM_GRID['dropout_rate']:
            for learning_rate in PARAM_GRID['learning_rate']:
                for batch_size in PARAM_GRID['batch_size']:
                    for n_epochs in PARAM_GRID['n_epochs']:
                        combo += 1
                        print(
                            f"  Combination {combo}/{total}: "
                            f"units={n_units}, dropout={dropout_rate}, "
                            f"lr={learning_rate}, batch={batch_size}, "
                            f"epochs={n_epochs}",
                            end="\r"
                        )

                        mean_val = cross_validate_lstm(
                            n_units, dropout_rate, learning_rate,
                            batch_size, n_epochs,
                            X_train_lstm, y_train_scaled,
                            n_splits=3
                        )

                        if mean_val < best_val_loss_grid:
                            best_val_loss_grid = mean_val
                            best_params_grid   = {
                                'n_units':       n_units,
                                'dropout_rate':  dropout_rate,
                                'learning_rate': learning_rate,
                                'batch_size':    batch_size,
                                'n_epochs':      n_epochs
                            }
                            print(
                                f"\n  New best — Val MSE: {best_val_loss_grid:.4f} "
                                f"| Params: {best_params_grid}"
                            )

    print(f"\n  Best Parameters (Lag {lag}): {best_params_grid}")
    print(f"  Best Validation MSE        : {best_val_loss_grid:.4f}")

    # Retrain final model on FULL training set with best parameters
    model_grid = build_lstm(
        best_params_grid['n_units'],
        best_params_grid['dropout_rate'],
        best_params_grid['learning_rate'],
        input_shape = X_train_lstm.shape[1:]
    )
    model_grid.fit(
        X_train_lstm, y_train_scaled,
        epochs     = best_params_grid['n_epochs'],
        batch_size = best_params_grid['batch_size'],
        verbose    = 0
    )

    # Predict and inverse scale
    y_pred_scaled = model_grid.predict(X_test_lstm, verbose=0).flatten()
    y_pred_inv    = scaler_y.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).flatten()

    # Compute and print per-lag metrics
    m = compute_metrics(y_test_inv, y_pred_inv)
    print_metrics(m, f"Grid Search LSTM — Lag {lag}")

    # Plot actual vs predicted
    plot_predictions(
        test_dates, y_test_inv, y_pred_inv,
        title=f"Grid Search LSTM — Lag {lag}: Actual vs Predicted Wind Speed"
    )

    grid_results.append({'lag': lag, 'best_params': best_params_grid, **m})

# Identify optimal lag for Grid Search
grid_summary  = pd.DataFrame(grid_results).set_index('lag')
best_grid_lag = grid_summary['rmse'].idxmin()
best_grid_row = grid_results[best_grid_lag - 1]

print("\n" + "=" * 70)
print("GRID SEARCH — SUMMARY ACROSS ALL LAGS")
print("=" * 70)
print(grid_summary[['mse', 'rmse', 'mae', 'mape', 'r2']].round(6))

# ── Optimal model box ────────────────────────────────────────────────────────
print_optimal_model_box(
    method_name = "Grid Search LSTM",
    optimal_lag = best_grid_lag,
    best_params = best_grid_row['best_params'],
    metrics     = {k: best_grid_row[k] for k in ['mse', 'rmse', 'mae', 'mape', 'r2']}
)



Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8: RANDOMIZED SEARCH — HYPERPARAMETERS × LAGS 1–14
# ─────────────────────────────────────────────────────────────────────────────
# For each lag (1–14), a random sample of hyperparameter combinations is
# evaluated using TimeSeriesSplit CV on the training set.
# The test set is NEVER accessed during the search.

print("\n" + "=" * 70)
print("SECTION 8: RANDOMIZED SEARCH HYPERPARAMETER TUNING")
print("=" * 70)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Number of random combinations to try per lag
N_ITER_RANDOM = 15

# Hyperparameter distributions for random sampling
RANDOM_PARAM_OPTIONS = {
    'n_units':       [32, 64, 96],
    'dropout_rate':  [0.1, 0.2, 0.3],
    'learning_rate': [1e-3, 3e-3, 1e-2],
    'batch_size':    [32, 64],
    'n_epochs':      [20, 30]
}


def sample_random_lstm_params(options, n_iter, seed):
    """
    Randomly sample n_iter LSTM hyperparameter combinations.

    Parameters:
        options : dict, lists of possible values per hyperparameter
        n_iter  : int, number of combinations to sample
        seed    : int, random seed for reproducibility

    Returns:
        list of dicts, sampled hyperparameter combinations
    """
    rng         = random.Random(seed)
    params_list = []
    for _ in range(n_iter):
        params_list.append({k: rng.choice(v) for k, v in options.items()})
    return params_list


random_results = []

for lag in range(1, 15):
    print(f"\n{'=' * 50}")
    print(f"  Randomized Search — Lag {lag}")
    print(f"{'=' * 50}")

    (X_train_lstm, X_test_lstm,
     y_train_scaled, y_test_scaled,
     y_test_inv, y_test_raw,
     test_dates, scaler_y) = prepare_lag_data(lag)

    sampled_params           = sample_random_lstm_params(
        RANDOM_PARAM_OPTIONS, N_ITER_RANDOM, SEED
    )
    best_val_loss_random     = float('inf')
    best_params_random       = None

    # Random search using TimeSeriesSplit CV on training data only
    for i, params in enumerate(sampled_params):
        print(f"  Trial {i + 1}/{N_ITER_RANDOM}: {params}", end="\r")

        mean_val = cross_validate_lstm(
            params['n_units'],
            params['dropout_rate'],
            params['learning_rate'],
            params['batch_size'],
            params['n_epochs'],
            X_train_lstm, y_train_scaled,
            n_splits=5
        )

        if mean_val < best_val_loss_random:
            best_val_loss_random = mean_val
            best_params_random   = params
            print(
                f"\n  New best — Val MSE: {best_val_loss_random:.4f} "
                f"| Params: {best_params_random}"
            )

    print(f"\n  Best Parameters (Lag {lag}): {best_params_random}")
    print(f"  Best Validation MSE        : {best_val_loss_random:.4f}")

    # Retrain final model on FULL training set with best parameters
    model_random = build_lstm(
        best_params_random['n_units'],
        best_params_random['dropout_rate'],
        best_params_random['learning_rate'],
        input_shape = X_train_lstm.shape[1:]
    )
    model_random.fit(
        X_train_lstm, y_train_scaled,
        epochs     = best_params_random['n_epochs'],
        batch_size = best_params_random['batch_size'],
        verbose    = 0
    )

    # Predict and inverse scale
    y_pred_scaled = model_random.predict(X_test_lstm, verbose=0).flatten()
    y_pred_inv    = scaler_y.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).flatten()

    # Compute and print per-lag metrics
    m = compute_metrics(y_test_inv, y_pred_inv)
    print_metrics(m, f"Randomized Search LSTM — Lag {lag}")

    # Plot actual vs predicted
    plot_predictions(
        test_dates, y_test_inv, y_pred_inv,
        title=f"Randomized Search LSTM — Lag {lag}: Actual vs Predicted Wind Speed"
    )

    random_results.append({'lag': lag, 'best_params': best_params_random, **m})

# Identify optimal lag for Randomized Search
random_summary  = pd.DataFrame(random_results).set_index('lag')
best_random_lag = random_summary['rmse'].idxmin()
best_random_row = random_results[best_random_lag - 1]

print("\n" + "=" * 70)
print("RANDOMIZED SEARCH — SUMMARY ACROSS ALL LAGS")
print("=" * 70)
print(random_summary[['mse', 'rmse', 'mae', 'mape', 'r2']].round(6))

# ── Optimal model box ────────────────────────────────────────────────────────
print_optimal_model_box(
    method_name = "Randomized Search LSTM",
    optimal_lag = best_random_lag,
    best_params = best_random_row['best_params'],
    metrics     = {k: best_random_row[k] for k in ['mse', 'rmse', 'mae', 'mape', 'r2']}
)



Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9: OPTUNA BAYESIAN HYPERPARAMETER OPTIMISATION
# ─────────────────────────────────────────────────────────────────────────────
# Optuna jointly optimises both the lag length (1–14) and the LSTM
# hyperparameters in a single study. The objective uses TimeSeriesSplit CV
# exclusively on the training set. The test set is NEVER accessed here.
#
# *** DATA LEAKAGE FIX ***
# The original Optuna code evaluated trials on X_test_scaled directly,
# meaning the test set drove the hyperparameter search — a textbook leakage
# issue. Fixed: all CV evaluation is now on training folds only.

print("\n" + "=" * 70)
print("SECTION 9: OPTUNA BAYESIAN OPTIMISATION")
print("=" * 70)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def objective_optuna(trial):
    """
    Optuna objective: jointly optimises lag length and LSTM hyperparameters.
    Evaluated using TimeSeriesSplit CV on the training set only.
    The test set is NEVER accessed inside this function.
    """
    # Suggest lag length as part of the optimisation
    lag = trial.suggest_int('lag', 1, 14)

    # Suggest LSTM hyperparameters
    n_units       = trial.suggest_int(  'n_units',       10,   128)
    dropout_rate  = trial.suggest_float('dropout_rate',  0.1,   0.4)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size    = trial.suggest_int(  'batch_size',    16,    64)
    n_epochs      = trial.suggest_int(  'n_epochs',      10,    50)

    # Prepare data for the suggested lag (training portion only)
    (X_train_lstm, _, y_train_scaled, _, _, _, _, _) = prepare_lag_data(lag)

    # Cross-validate on training data — test set is never touched
    return cross_validate_lstm(
        n_units, dropout_rate, learning_rate,
        batch_size, n_epochs,
        X_train_lstm, y_train_scaled,
        n_splits=3
    )


# Run Optuna study — minimise validation MSE on training data only
study = optuna.create_study(
    direction = 'minimize',
    sampler   = TPESampler(seed=SEED)
)
study.optimize(objective_optuna, n_trials=50)

# Extract best lag and hyperparameters from the study
best_trial_params = dict(study.best_trial.params)
best_lag_optuna   = best_trial_params.pop('lag')   # Remove lag from params dict
best_params_optuna = best_trial_params

print(f"\n→ Optimal Lag (Optuna)  : {best_lag_optuna}")
print(f"  Best Params           : {best_params_optuna}")
print(f"  Best Validation MSE   : {study.best_trial.value:.6f}")

# Prepare data for the optimal lag
(X_train_lstm, X_test_lstm,
 y_train_scaled, y_test_scaled,
 y_test_inv, y_test_raw,
 test_dates, scaler_y) = prepare_lag_data(best_lag_optuna)

# Retrain the final model on the FULL training set with Optuna best parameters
model_optuna = build_lstm(
    best_params_optuna['n_units'],
    best_params_optuna['dropout_rate'],
    best_params_optuna['learning_rate'],
    input_shape = X_train_lstm.shape[1:]
)
model_optuna.fit(
    X_train_lstm, y_train_scaled,
    epochs     = best_params_optuna['n_epochs'],
    batch_size = best_params_optuna['batch_size'],
    verbose    = 1
)

# Predict and inverse scale
y_pred_scaled_optuna = model_optuna.predict(X_test_lstm, verbose=0).flatten()
y_pred_inv_optuna    = scaler_y.inverse_transform(
    y_pred_scaled_optuna.reshape(-1, 1)
).flatten()

# Compute and print metrics
optuna_metrics = compute_metrics(y_test_inv, y_pred_inv_optuna)
print_metrics(optuna_metrics, f"Optuna LSTM — Best Lag {best_lag_optuna}")

# Plot actual vs predicted
plot_predictions(
    test_dates, y_test_inv, y_pred_inv_optuna,
    title=f"Optuna LSTM — Lag {best_lag_optuna}: Actual vs Predicted Wind Speed"
)

# ── Optimal model box ────────────────────────────────────────────────────────
print_optimal_model_box(
    method_name = "Optuna LSTM",
    optimal_lag = best_lag_optuna,
    best_params = best_params_optuna,
    metrics     = optuna_metrics
)



In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10: FINAL COMPARATIVE RESULTS TABLE
# ─────────────────────────────────────────────────────────────────────────────
# Each method is represented by its optimal lag and corresponding test metrics.

print("\n" + "=" * 70)
print("SECTION 10: FINAL COMPARATIVE RESULTS")
print("=" * 70)

grid_best   = {k: best_grid_row[k]   for k in ['mse', 'rmse', 'mae', 'mape', 'r2']}
random_best = {k: best_random_row[k] for k in ['mse', 'rmse', 'mae', 'mape', 'r2']}

comparison_df = pd.DataFrame({
    "Method":        ["Original Model",      "Grid Search",
                      "Randomized Search",   "Optuna"],
    "Optimal Lag":   [best_orig_lag,          best_grid_lag,
                      best_random_lag,        best_lag_optuna],
    "Test MSE":      [best_orig_row['mse'],   grid_best['mse'],
                      random_best['mse'],     optuna_metrics['mse']],
    "Test RMSE":     [best_orig_row['rmse'],  grid_best['rmse'],
                      random_best['rmse'],    optuna_metrics['rmse']],
    "Test MAE":      [best_orig_row['mae'],   grid_best['mae'],
                      random_best['mae'],     optuna_metrics['mae']],
    "Test MAPE (%)": [best_orig_row['mape'],  grid_best['mape'],
                      random_best['mape'],    optuna_metrics['mape']],
    "Test R²":       [best_orig_row['r2'],    grid_best['r2'],
                      random_best['r2'],      optuna_metrics['r2']],
})

print("\nOptimal Lag and Test Set Performance by Method:")
print(comparison_df.to_string(index=False))

# Identify the overall best method by Test RMSE
best_method_idx = comparison_df['Test RMSE'].idxmin()
best_method_row = comparison_df.iloc[best_method_idx]

print(f"\n{'=' * 70}")
print(f"  OVERALL BEST METHOD  : {best_method_row['Method']}")
print(f"  Optimal Lag          : {best_method_row['Optimal Lag']}")
print(f"  Test MSE             : {best_method_row['Test MSE']:.6f}")
print(f"  Test RMSE            : {best_method_row['Test RMSE']:.6f}")
print(f"  Test MAE             : {best_method_row['Test MAE']:.6f}")
print(f"  Test MAPE            : {best_method_row['Test MAPE (%)']:.6f} %")
print(f"  Test R²              : {best_method_row['Test R²']:.6f}")
print(f"{'=' * 70}")

# Save the comparison table to CSV
comparison_df.to_csv("LSTM_LagOptimisation_Results.csv", index=False)
print("\nResults saved to LSTM_LagOptimisation_Results.csv")
